# 📐 Linear Algebra for AI

---

## 1. Overview

Linear algebra is the **mathematical language of AI**. Every neural network forward pass is a sequence of matrix multiplications. Embeddings are vectors. Attention scores are dot products. This notebook builds the linear algebra intuition you need to understand — and debug — AI systems.

We'll go from basic vectors to eigendecomposition and SVD, with every concept tied to a concrete AI application.

## 2. Learning Objectives

By the end of this notebook, you will be able to:

- Create and manipulate vectors, matrices, and tensors with NumPy
- Perform dot products and explain their role in similarity search
- Execute matrix multiplication and understand its role in neural networks
- Compute and interpret eigenvalues/eigenvectors for PCA
- Apply SVD for dimensionality reduction
- Visualize linear transformations geometrically

## 3. Imports

In [ ]:
import numpy as np                # Core numerical computing — version 1.26+
import matplotlib.pyplot as plt   # Visualization
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D

print(f"NumPy version: {np.__version__}")

## 4. Configuration

In [ ]:
# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.style.use('seaborn-v0_8-whitegrid')

## 5. Theory & Implementation

### 5.1 Scalars, Vectors, Matrices, and Tensors

| Object | Rank | Example in AI |
|--------|------|---------------|
| Scalar | 0 | A single weight, a loss value |
| Vector | 1 | An embedding (e.g., a 768-d word embedding) |
| Matrix | 2 | A weight matrix in a linear layer |
| Tensor | 3+ | A batch of images: `(batch, height, width, channels)` |

$$\text{Scalar: } x \in \mathbb{R}, \quad \text{Vector: } \mathbf{v} \in \mathbb{R}^n, \quad \text{Matrix: } \mathbf{M} \in \mathbb{R}^{m \times n}$$

In [ ]:
# --- Scalars ---
learning_rate = 0.001  # A scalar — one number
loss_value = np.float64(2.345)
print(f"Scalar (learning rate): {learning_rate}")
print(f"Scalar (loss): {loss_value}, ndim={loss_value.ndim}")

# --- Vectors ---
# A word embedding (simplified to 5 dimensions)
embedding_king = np.array([0.9, 0.2, -0.5, 0.8, 0.1])
embedding_queen = np.array([0.85, 0.3, -0.4, 0.75, 0.6])
print(f"\nVector (king embedding): {embedding_king}, shape={embedding_king.shape}")
print(f"Vector (queen embedding): {embedding_queen}, shape={embedding_queen.shape}")

# --- Matrices ---
# A weight matrix: maps 5-dim input to 3-dim output (like a neural network layer)
weight_matrix = np.random.randn(3, 5)
print(f"\nWeight matrix:\n{weight_matrix}")
print(f"Shape: {weight_matrix.shape} — maps 5D → 3D")

# --- Tensors (3D+) ---
# A batch of 4 "images", each 8x8 pixels, 3 color channels
image_batch = np.random.rand(4, 8, 8, 3)
print(f"\n3D+ Tensor (image batch): shape={image_batch.shape}")
print(f"  → {image_batch.shape[0]} images, {image_batch.shape[1]}x{image_batch.shape[2]} pixels, {image_batch.shape[3]} channels")

### 5.2 The Dot Product — The Heart of Similarity

The dot product measures how **similar** two vectors are. It's the core operation behind:
- **Attention scores** in Transformers
- **Cosine similarity** in semantic search
- **Neuron activation** (input · weights)

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = |\mathbf{a}| |\mathbf{b}| \cos\theta$$

When vectors point in the same direction → large positive dot product → **similar**.  
When perpendicular → dot product = 0 → **unrelated**.  
When opposite → large negative dot product → **dissimilar**.

In [ ]:
# Dot product — the fundamental similarity measure
similarity = np.dot(embedding_king, embedding_queen)
print(f"Dot product (king · queen): {similarity:.4f}")

# Cosine similarity — normalized dot product (range: -1 to 1)
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

cos_sim = cosine_similarity(embedding_king, embedding_queen)
print(f"Cosine similarity (king, queen): {cos_sim:.4f}")

# Compare with a very different word
embedding_car = np.array([-0.3, 0.7, 0.9, -0.2, -0.5])
print(f"Cosine similarity (king, car): {cosine_similarity(embedding_king, embedding_car):.4f}")
print(f"\n→ 'king' is much more similar to 'queen' than to 'car' — as expected!")

In [ ]:
# Visualize dot product / cosine similarity in 2D
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cases = [
    (np.array([1, 0.5]), np.array([0.8, 0.4]), "Similar (θ small)"),
    (np.array([1, 0]), np.array([0, 1]), "Perpendicular (θ = 90°)"),
    (np.array([1, 0.5]), np.array([-0.8, -0.4]), "Opposite (θ ≈ 180°)"),
]

for ax, (a, b, title) in zip(axes, cases):
    ax.quiver(0, 0, a[0], a[1], angles='xy', scale_units='xy', scale=1,
              color='#e74c3c', linewidth=2, label=f'a = {a}')
    ax.quiver(0, 0, b[0], b[1], angles='xy', scale_units='xy', scale=1,
              color='#3498db', linewidth=2, label=f'b = {b}')
    cos = cosine_similarity(a, b)
    ax.set_title(f"{title}\ncos(θ) = {cos:.2f}", fontweight='bold')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1, 1.2)
    ax.set_aspect('equal')
    ax.legend(fontsize=9)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)

plt.suptitle('Dot Product & Cosine Similarity — Visual Intuition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Matrix Multiplication — The Core of Neural Networks

Every linear layer in a neural network is a matrix multiplication:

$$\mathbf{y} = \mathbf{W} \mathbf{x} + \mathbf{b}$$

Where:
- $\mathbf{x} \in \mathbb{R}^n$ is the input
- $\mathbf{W} \in \mathbb{R}^{m \times n}$ is the weight matrix
- $\mathbf{b} \in \mathbb{R}^m$ is the bias
- $\mathbf{y} \in \mathbb{R}^m$ is the output

**Rule:** For $\mathbf{A}_{m \times k} \cdot \mathbf{B}_{k \times n}$, the inner dimensions must match → result is $m \times n$.

In [ ]:
# Simulate a neural network layer: 5-dim input → 3-dim output
W = np.random.randn(3, 5)   # Weight matrix (3 output neurons, 5 inputs each)
b = np.random.randn(3)      # Bias vector
x = embedding_king           # Our 5-dim input

# Forward pass: y = Wx + b
y = W @ x + b  # '@' is the matrix multiplication operator

print(f"Input x:  shape={x.shape}  → {x}")
print(f"Weight W: shape={W.shape}")
print(f"Bias b:   shape={b.shape}  → {b}")
print(f"Output y: shape={y.shape}  → {y}")
print(f"\n→ This is exactly what nn.Linear(5, 3) does in PyTorch!")

In [ ]:
# Batch matrix multiplication — processing multiple inputs at once
# In practice, we process batches of data, not single vectors
batch_size = 4
X_batch = np.random.randn(batch_size, 5)  # 4 inputs, each 5-dimensional

# Forward pass on the whole batch: Y = XW^T + b
Y_batch = X_batch @ W.T + b

print(f"Input batch:  shape={X_batch.shape}  (batch_size × input_dim)")
print(f"Weight matrix: shape={W.shape}        (output_dim × input_dim)")
print(f"Output batch: shape={Y_batch.shape}   (batch_size × output_dim)")
print(f"\n→ 4 inputs processed simultaneously — this is why GPUs are fast!")

### 5.4 Matrix Transformations — What Weight Matrices Actually Do

A matrix multiplication is a **linear transformation** — it can rotate, scale, shear, or project data. Understanding this geometrically helps debug model behavior.

In [ ]:
# Visualize how different matrices transform a unit square
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

# Unit square vertices
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]]).T

transforms = [
    (np.array([[2, 0], [0, 2]]), "Scaling (2x)"),
    (np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
               [np.sin(np.pi/4), np.cos(np.pi/4)]]), "Rotation (45°)"),
    (np.array([[1, 0.5], [0, 1]]), "Shearing"),
    (np.array([[1, 0], [0, 0]]), "Projection (→ x-axis)"),
]

for ax, (M, title) in zip(axes, transforms):
    transformed = M @ square
    ax.fill(square[0], square[1], alpha=0.3, color='#3498db', label='Original')
    ax.fill(transformed[0], transformed[1], alpha=0.3, color='#e74c3c', label='Transformed')
    ax.plot(square[0], square[1], 'b-', linewidth=1.5)
    ax.plot(transformed[0], transformed[1], 'r-', linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(-1, 3)
    ax.set_ylim(-1, 3)
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.axhline(y=0, color='k', linewidth=0.3)
    ax.axvline(x=0, color='k', linewidth=0.3)

plt.suptitle('Matrix Transformations — What Weight Matrices Do', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.5 Norms — Measuring Vector Magnitude

Norms measure the "size" of a vector. Critical for:
- **Normalizing embeddings** before cosine similarity
- **Gradient clipping** (preventing exploding gradients)
- **Regularization** (L1, L2 penalties on weights)

| Norm | Formula | Use in AI |
|------|---------|----------|
| L1 (Manhattan) | $\|\mathbf{x}\|_1 = \sum |x_i|$ | Sparse regularization (Lasso) |
| L2 (Euclidean) | $\|\mathbf{x}\|_2 = \sqrt{\sum x_i^2}$ | Weight decay, gradient clipping |
| L∞ (Max) | $\|\mathbf{x}\|_\infty = \max|x_i|$ | Adversarial robustness |

In [ ]:
v = np.array([3.0, -4.0, 2.0, -1.0, 5.0])

l1_norm = np.linalg.norm(v, ord=1)
l2_norm = np.linalg.norm(v, ord=2)
linf_norm = np.linalg.norm(v, ord=np.inf)

print(f"Vector: {v}")
print(f"L1 norm (Manhattan):  {l1_norm:.4f}")
print(f"L2 norm (Euclidean):  {l2_norm:.4f}")
print(f"L∞ norm (Max):        {linf_norm:.4f}")

# Normalize a vector (unit vector) — used before cosine similarity
v_normalized = v / np.linalg.norm(v)
print(f"\nNormalized: {v_normalized}")
print(f"Norm of normalized vector: {np.linalg.norm(v_normalized):.4f} (should be 1.0)")

### 5.6 Eigenvalues & Eigenvectors — Understanding PCA

For a square matrix $\mathbf{A}$, an eigenvector $\mathbf{v}$ satisfies:

$$\mathbf{A} \mathbf{v} = \lambda \mathbf{v}$$

The matrix only **scales** the eigenvector by $\lambda$ (the eigenvalue), without changing its direction.

**Why this matters for AI:**
- **PCA** finds the eigenvectors of the covariance matrix → directions of maximum variance
- Used to visualize high-dimensional embeddings in 2D/3D
- Helps understand what a linear layer "focuses on"

In [ ]:
# Create a covariance-like matrix (symmetric, positive semi-definite)
A = np.array([[4, 2],
              [2, 3]])

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Matrix A:\n{A}\n")
print(f"Eigenvalues: {eigenvalues}")
print(f"Eigenvectors (columns):\n{eigenvectors}\n")

# Verify: A @ v = λ * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    Av = A @ v
    lam_v = lam * v
    print(f"Eigenvector {i+1}: A·v = {Av}, λ·v = {lam_v}, match = {np.allclose(Av, lam_v)}")

In [ ]:
# Visualize eigenvectors on 2D data (PCA intuition)
# Generate correlated 2D data
mean = [0, 0]
cov = [[4, 2], [2, 3]]  # Same matrix as above
data = np.random.multivariate_normal(mean, cov, 300)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(data[:, 0], data[:, 1], alpha=0.4, s=20, color='#3498db')

# Plot eigenvectors scaled by eigenvalues
colors = ['#e74c3c', '#2ecc71']
for i in range(2):
    v = eigenvectors[:, i] * eigenvalues[i]  # Scale by eigenvalue
    ax.annotate('', xy=v, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=3))
    ax.text(v[0]*1.1, v[1]*1.1, f'PC{i+1} (λ={eigenvalues[i]:.2f})',
            fontsize=12, fontweight='bold', color=colors[i])

ax.set_title('PCA Intuition: Eigenvectors Show Directions of Maximum Variance',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_aspect('equal')
ax.set_xlim(-7, 7)
ax.set_ylim(-7, 7)
plt.tight_layout()
plt.show()

### 5.7 Singular Value Decomposition (SVD)

SVD factorizes **any** matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$ as:

$$\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$$

Where:
- $\mathbf{U}$ — left singular vectors (m × m)
- $\boldsymbol{\Sigma}$ — diagonal matrix of singular values (m × n)
- $\mathbf{V}^T$ — right singular vectors (n × n)

**AI Applications:**
- **Dimensionality reduction** (truncated SVD → LSA in NLP)
- **LoRA** — Low-Rank Adaptation approximates weight updates with low-rank matrices
- **Image compression**

In [ ]:
# SVD for image compression — a compelling visual demonstration
# Create a simple grayscale "image" (a gradient pattern)
x_grid = np.linspace(-3, 3, 100)
y_grid = np.linspace(-3, 3, 80)
X, Y = np.meshgrid(x_grid, y_grid)
image = np.sin(X) * np.cos(Y) + 0.5 * np.sin(2*X + Y)

# Perform SVD
U, S, Vt = np.linalg.svd(image, full_matrices=False)

# Reconstruct with different numbers of singular values
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
ranks = [1, 5, 10, 20, len(S)]

for ax, k in zip(axes, ranks):
    # Truncated SVD reconstruction
    reconstructed = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    ax.imshow(reconstructed, cmap='viridis')
    compression = 100 * (1 - k * (80 + 100 + 1) / (80 * 100))
    ax.set_title(f'Rank {k}\n({compression:.0f}% compression)', fontweight='bold')
    ax.axis('off')

plt.suptitle('SVD Image Compression — Same Principle as LoRA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show singular value decay
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(S)), S, color='#3498db', alpha=0.7)
ax.set_xlabel('Singular Value Index')
ax.set_ylabel('Magnitude')
ax.set_title('Singular Values — Most Information is in the First Few', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Evaluation — Putting It All Together

Let's build a **mini semantic search engine** using only linear algebra:

In [ ]:
# Mini semantic search using cosine similarity
# Simulated word embeddings (in practice, these come from models like BERT)
vocabulary = {
    "king":     np.array([0.9, 0.2, -0.5, 0.8, 0.1, 0.3]),
    "queen":    np.array([0.85, 0.3, -0.4, 0.75, 0.6, 0.35]),
    "prince":   np.array([0.7, 0.15, -0.45, 0.65, 0.05, 0.25]),
    "princess": np.array([0.65, 0.25, -0.35, 0.6, 0.55, 0.3]),
    "man":      np.array([0.5, 0.1, -0.3, 0.4, -0.1, 0.2]),
    "woman":    np.array([0.45, 0.2, -0.2, 0.35, 0.5, 0.25]),
    "car":      np.array([-0.3, 0.7, 0.9, -0.2, -0.5, 0.8]),
    "truck":    np.array([-0.25, 0.65, 0.85, -0.15, -0.45, 0.75]),
    "bicycle":  np.array([-0.2, 0.6, 0.7, -0.1, -0.3, 0.6]),
}

def search(query_word: str, top_k: int = 5) -> list[tuple[str, float]]:
    """Find the most similar words to the query using cosine similarity."""
    query_vec = vocabulary[query_word]
    similarities = []
    for word, vec in vocabulary.items():
        if word != query_word:
            sim = cosine_similarity(query_vec, vec)
            similarities.append((word, sim))
    return sorted(similarities, key=lambda x: x[1], reverse=True)[:top_k]

# Search for words similar to "king"
print("🔍 Words most similar to 'king':")
for word, sim in search("king"):
    bar = '█' * int(sim * 30)
    print(f"  {word:12s} {sim:.4f} {bar}")

print("\n🔍 Words most similar to 'car':")
for word, sim in search("car"):
    bar = '█' * int(max(0, sim * 30))
    print(f"  {word:12s} {sim:.4f} {bar}")

In [ ]:
# The famous word analogy: king - man + woman ≈ queen
result = vocabulary["king"] - vocabulary["man"] + vocabulary["woman"]

print("🧮 king - man + woman = ?\n")
for word, vec in vocabulary.items():
    sim = cosine_similarity(result, vec)
    bar = '█' * int(max(0, sim * 30))
    print(f"  {word:12s} {sim:.4f} {bar}")

best_match = max(vocabulary.items(), key=lambda x: cosine_similarity(result, x[1]))
print(f"\n✅ Best match: '{best_match[0]}' — Linear algebra captures meaning!")

## 8. Exercises

### Exercise 1: Matrix Shapes
Given a neural network layer with 768 input features and 256 output features processing a batch of 32 samples:
- What is the shape of the weight matrix?
- What is the shape of the input batch?
- What is the shape of the output batch?

In [ ]:
# Exercise 1 — Fill in the shapes
input_dim = 768
output_dim = 256
batch_size = 32

W_ex = np.random.randn(output_dim, input_dim)  # (256, 768)
X_ex = np.random.randn(batch_size, input_dim)   # (32, 768)
Y_ex = X_ex @ W_ex.T                             # (32, 256)

print(f"Weight matrix shape: {W_ex.shape}")
print(f"Input batch shape:   {X_ex.shape}")
print(f"Output batch shape:  {Y_ex.shape}")

### Exercise 2: Build a Similarity Matrix
Compute the pairwise cosine similarity between all words in our vocabulary.

In [ ]:
# Exercise 2 — Similarity heatmap
words = list(vocabulary.keys())
n = len(words)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(vocabulary[words[i]], vocabulary[words[j]])

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(words, rotation=45, ha='right')
ax.set_yticklabels(words)
plt.colorbar(im, label='Cosine Similarity')
ax.set_title('Word Similarity Matrix', fontweight='bold', fontsize=13)

# Add values to cells
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

## 9. Challenge Problems

### Challenge 1: Implement PCA from Scratch
Using only NumPy (no scikit-learn), implement PCA:
1. Center the data (subtract mean)
2. Compute the covariance matrix
3. Compute eigenvectors/eigenvalues
4. Project the data onto the top-k eigenvectors

### Challenge 2: Power Iteration
Implement the power iteration algorithm to find the dominant eigenvalue/eigenvector of a matrix. This is how Google's PageRank originally worked.

### Challenge 3: Low-Rank Approximation
Load a real image with matplotlib, convert to grayscale, and use truncated SVD to compress it. Plot the reconstruction error vs. number of singular values.

In [ ]:
# Challenge 1 starter: PCA from scratch
def pca(X: np.ndarray, n_components: int = 2) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Principal Component Analysis from scratch.
    
    Args:
        X: Data matrix of shape (n_samples, n_features).
        n_components: Number of principal components to keep.
    
    Returns:
        X_projected: Projected data of shape (n_samples, n_components).
        components: Principal components (eigenvectors).
        explained_variance: Variance explained by each component.
    """
    # Step 1: Center the data
    X_centered = X - X.mean(axis=0)
    
    # Step 2: Compute covariance matrix
    cov_matrix = np.cov(X_centered, rowvar=False)
    
    # Step 3: Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # Step 4: Sort by eigenvalue (descending)
    sorted_idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[sorted_idx]
    eigenvectors = eigenvectors[:, sorted_idx]
    
    # Step 5: Select top-k components and project
    components = eigenvectors[:, :n_components]
    X_projected = X_centered @ components
    
    explained_variance = eigenvalues[:n_components] / eigenvalues.sum()
    
    return X_projected, components, explained_variance

# Test on synthetic 5D data
data_5d = np.random.randn(200, 5) @ np.random.randn(5, 5)
projected, components, var_explained = pca(data_5d, n_components=2)

print(f"Original shape: {data_5d.shape}")
print(f"Projected shape: {projected.shape}")
print(f"Variance explained: PC1={var_explained[0]:.2%}, PC2={var_explained[1]:.2%}")

plt.figure(figsize=(8, 6))
plt.scatter(projected[:, 0], projected[:, 1], alpha=0.5, s=20, c='#3498db')
plt.xlabel(f'PC1 ({var_explained[0]:.1%} variance)')
plt.ylabel(f'PC2 ({var_explained[1]:.1%} variance)')
plt.title('PCA from Scratch — 5D → 2D Projection', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [3Blue1Brown — Essence of Linear Algebra](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab) | 🎥 Video | The best visual introduction to linear algebra |
| [Mathematics for Machine Learning](https://mml-book.github.io/) | 📘 Book | Free textbook covering LA for ML |
| [NumPy Linear Algebra docs](https://numpy.org/doc/stable/reference/routines.linalg.html) | 📖 Docs | Official NumPy linalg reference |
| [Stanford CS229 — Linear Algebra Review](https://cs229.stanford.edu/section/cs229-linalg.pdf) | 📄 Notes | Concise review for ML contexts |

---

**Next:** [02 — Calculus & Optimization →](02-calculus-and-optimization.ipynb)